## Environment setup
- mujoco: https://di-engine-docs.readthedocs.io/en/latest/13_envs/mujoco.html

In [1]:
import os
os.chdir("..")

from utils.utils import notebook_line_magic
notebook_line_magic()

Line Magic Set


In [2]:
from utils.utils import set_ld_library_path
set_ld_library_path()

MUJOCO_GL = osmesa
MUJOCO_PY_MUJOCO_PATH = /home/azm0269@auburn.edu/.mujoco/mujoco210
LD_LIBRARY_PATH entries:
   /home/azm0269@auburn.edu/.mujoco/mujoco210/bin
   /home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/mujoco_py/generated/_pyxbld_2.1.2.14_310_linuxcpuextensionbuilder/lib.linux-x86_64-cpython-310/mujoco_py


In [3]:
# !pip uninstall -y mujoco-py
# !pip install --no-cache-dir "gym[mujoco]==0.25.1" "mujoco-py<2.2,>=2.1" "cython<3" "numpy<2"

In [3]:
# import importlib.util, glob, os
# root = importlib.util.find_spec("mujoco_py").submodule_search_locations[0]
# print(glob.glob(root + "/**/libglewosmesa.so", recursive=True))

In [4]:
# !pip uninstall -y mujoco-py
# !pip install --no-cache-dir "mujoco-py==2.1.2.14" "cython<3" "numpy<2"

In [3]:
import gym
env = gym.make('halfcheetah-expert-v2')
obs = env.reset()
# obs[0].shape

NameNotFound: Environment halfcheetah-expert doesn't exist. Did you mean: `HalfCheetah`?

In [12]:
obs

array([ 0.0485011 ,  0.03007728,  0.03600819,  0.00793457,  0.0825652 ,
       -0.00558539,  0.0524024 ,  0.05422565, -0.169217  ,  0.00232832,
       -0.05308315, -0.01990058, -0.0019455 ,  0.04047802,  0.0961307 ,
        0.13179492, -0.23637308])

In [8]:
# !pip list | grep gym

## Store Video

In [10]:
from easydict import EasyDict
from dizoo.mujoco.envs import MujocoEnv

config = MujocoEnv.default_config()
config.env_id = "Hopper-v4"
env = MujocoEnv(config)
env.enable_save_replay(replay_path="./video/mujoco")
obs = env.reset()
while True:
    action = env.random_action()
    timestep = env.step(action)
    if timestep.done:
        print("Episode is over, eval episode retrun is: {}".format(timestep.info['eval_episode_return']))
        break

Episode is over, eval episode retrun is: 25.09187705112608


### Run Hoppper using SAC

In [15]:
%%writefile baselines/sac_hopper_v4.py
from easydict import EasyDict

hopper_sac_config = dict(
    exp_name='hopper_sac_seed0',
    env=dict(
        env_id='Hopper-v4',
        norm_obs=dict(use_norm=False, ),
        norm_reward=dict(use_norm=False, ),
        collector_env_num=1,
        evaluator_env_num=8,
        n_evaluator_episode=8,
        stop_value=6000,
    ),
    policy=dict(
        cuda=True,
        random_collect_size=10000,
        model=dict(
            obs_shape=11,
            action_shape=3,
            twin_critic=True,
            action_space='reparameterization',
            actor_head_hidden_size=256,
            critic_head_hidden_size=256,
        ),
        learn=dict(
            update_per_collect=1,
            batch_size=256,
            learning_rate_q=1e-3,
            learning_rate_policy=1e-3,
            learning_rate_alpha=3e-4,
            ignore_done=False,
            target_theta=0.005,
            discount_factor=0.99,
            alpha=0.2,
            reparameterization=True,
            auto_alpha=False,
        ),
        collect=dict(
            n_sample=1,
            unroll_len=1,
        ),
        command=dict(),
        eval=dict(),
        other=dict(replay_buffer=dict(replay_buffer_size=1000000, ), ),
    ),
)

hopper_sac_config = EasyDict(hopper_sac_config)
main_config = hopper_sac_config

hopper_sac_create_config = dict(
    env=dict(
        type='mujoco',
        import_names=['dizoo.mujoco.envs.mujoco_env'],
    ),
    env_manager=dict(type='subprocess'),
    policy=dict(
        type='sac',
        import_names=['ding.policy.sac'],
    ),
    replay_buffer=dict(type='naive', ),
)
hopper_sac_create_config = EasyDict(hopper_sac_create_config)
create_config = hopper_sac_create_config

if __name__ == "__main__":
    # or you can enter `ding -m serial -c hopper_sac_config.py -s 0`
    from ding.entry import serial_pipeline
    serial_pipeline([main_config, create_config], seed=0)

Overwriting baselines/sac_hopper_v4.py


### Flow
1. Set up a set of environments.
2. Collect offline data and store it.
3. Run offline baselines on the collected data.